# 🌿 PROP-DEOCCNET — STUDI ABLASI (M0–M3) SAJA
**Companion notebook untuk `kaggle_full_train.ipynb`** — Dhea Anggita, Magister Ilmu Komputer UGM 2026

---

### 📋 Kenapa notebook terpisah?
Notebook `kaggle_full_train.ipynb` menjalankan **training utama (80 epoch) + studi ablasi (4×20 epoch)** dalam satu sesi (~9–10,5 jam total). Ini rawan terpotong oleh batas durasi sesi atau kuota GPU mingguan Kaggle sebelum studi ablasi (bagian paling penting untuk pembuktian H1/H2/H3 di tesis) selesai.

Notebook ini **hanya** menjalankan studi ablasi M0–M3 (Tesis Bab III, Tabel 3.1) — tanpa training utama 80 epoch. Gunakan ini **setelah** `kaggle_full_train.ipynb` sudah pernah dijalankan sekali dan menghasilkan `best.pth` + metrik test set yang baik (studi ablasi tidak bergantung pada checkpoint itu — setiap varian M0–M3 dilatih dari nol secara independen — tapi tidak ada gunanya mengulang training utama kalau hasilnya sudah bagus).

**Alur di notebook ini:**
* **Step 1**: Setup GPU
* **Step 2**: Clone repo & konfigurasi dataset
* **Step 3**: Jalankan studi ablasi M0–M3 (~5–6 jam untuk 4×20 epoch di T4) + tampilkan hasil

💡 Kalau sesi ini terputus di tengah jalan, cukup jalankan ulang notebook ini di sesi baru — asalkan `checkpoints/ablation_results.json` dari sesi sebelumnya ikut terbawa (lihat catatan di akhir notebook), varian yang sudah selesai tidak akan dilatih ulang.

## Step 1 — Setup Lingkungan & Akselerasi GPU

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Cek GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[{'✓' if torch.cuda.is_available() else '✗'}] Device aktif: {device}")
if torch.cuda.is_available():
    print(f"    GPU Name : {torch.cuda.get_device_name(0)}")
    print(f"    VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set styling grafik matplotlib agar standar publikasi ilmiah
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.0


## Step 2 — Clone Repository & Konfigurasi Dataset
Sama seperti di `kaggle_full_train.ipynb`: clone branch `roboflow` (atau tarik commit terbaru kalau folder sudah ada), lalu deteksi & set path dataset COCO di `training/config_train.yaml`.

In [ ]:
import yaml
import shutil

# Root directory proyek
WORK_DIR = "/kaggle/working/labeling-daun-itoh" if os.path.exists("/kaggle") else "."
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Clone repo jika di Kaggle (atau tarik commit terbaru jika folder sudah ada
# dari sesi sebelumnya, supaya fix terbaru di branch roboflow selalu terpakai)
REPO_URL = "https://github.com/cemilick/anotasi-daun.git"
BRANCH = "roboflow"

if os.path.exists("/kaggle"):
    if not os.path.exists(WORK_DIR):
        !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
    else:
        print(f"[INFO] {WORK_DIR} sudah ada, menarik commit terbaru dari branch '{BRANCH}'...")
        !git -C {WORK_DIR} fetch origin {BRANCH}
        !git -C {WORK_DIR} checkout {BRANCH}
        !git -C {WORK_DIR} reset --hard origin/{BRANCH}
    os.chdir(WORK_DIR)
elif os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)

print(f"[INFO] Working Directory: {os.getcwd()}")
if os.path.exists("/kaggle"):
    commit_info = !git log -1 --format="%h %ci %s"
    print(f"[INFO] Branch aktif    : {BRANCH}")
    print(f"[INFO] Commit terkini  : {commit_info[0] if commit_info else '?'}")

# ── 1. PENGATURAN PATH DATASET (SET LANGSUNG DI SINI) ──────────────────────────
# Jika Anda di Kaggle dan nama folder dataset berbeda, sesuaikan DATASET_DIR di bawah ini:
DATASET_DIR = "/kaggle/input/daun-kelengkeng-itoh" if os.path.exists("/kaggle") else "output/coco"

# Auto-detect jika folder di Kaggle bernama lain (misal: /kaggle/input/labeling-daun-itoh-v2)
import glob
if os.path.exists("/kaggle/input") and not os.path.exists(DATASET_DIR):
    found_dirs = glob.glob("/kaggle/input/*")
    if found_dirs:
        DATASET_DIR = found_dirs[0]
        print(f"[AUTO-DETECT] Folder dataset Kaggle ditemukan di: {DATASET_DIR}")

# ── 2. LOAD & UPDATE CONFIG_TRAIN.YAML ─────────────────────────────────────────
CONFIG_PATH = "training/config_train.yaml"
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
        
    def find_coco_json(default_path, split_names):
        # 1. Cek langsung di bawah DATASET_DIR
        for sname in split_names:
            p = f"{DATASET_DIR}/{sname}/_annotations.coco.json"
            if os.path.exists(p): return p
            p_alt = f"{DATASET_DIR}/{sname}.json"
            if os.path.exists(p_alt): return p_alt
        # 2. Cek glob recursive di DATASET_DIR & /kaggle/input
        for sname in split_names:
            found = glob.glob(f"{DATASET_DIR}/**/{sname}*json", recursive=True)
            if not found and os.path.exists("/kaggle/input"):
                found = glob.glob(f"/kaggle/input/**/*{sname}*json", recursive=True)
            if found: return found[0]
        return default_path

    cfg["train_json"] = find_coco_json(cfg.get("train_json"), ["train", "training"])
    cfg["val_json"]   = find_coco_json(cfg.get("val_json"), ["valid", "val", "validation"])
    cfg["test_json"]  = find_coco_json(cfg.get("test_json"), ["test", "testing"])
    
    # PENTING: Simpan kembali (overwrite) ke config_train.yaml di disk!
    # Sehingga saat train.py / visualize.py dipanggil, path yang dibaca dari disk sudah benar!
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, default_flow_style=False)
        
    print("[✓] Path Dataset berhasil diset & disimpan kembali ke config_train.yaml:")
    print(f"    Train JSON       : {cfg['train_json']} [{'✓ Ada' if os.path.exists(str(cfg['train_json'])) else '✗ TIDAK DITEMUKAN'}]")
    print(f"    Val JSON         : {cfg['val_json']} [{'✓ Ada' if os.path.exists(str(cfg['val_json'])) else '✗ TIDAK DITEMUKAN'}]")
    print(f"    Test JSON        : {cfg['test_json']} [{'✓ Ada' if os.path.exists(str(cfg['test_json'])) else '✗ TIDAK DITEMUKAN'}]")
    print(f"    Backbone         : {cfg.get('backbone', 'resnet101')}")
    print(f"    ASPP Rates       : {cfg.get('aspp_rates', [6, 12, 18, 24])}")
    print(f"    Boundary Head    : {cfg.get('use_boundary_head', True)}")
    print(f"    Epochs           : {cfg.get('epochs', 50)}")
    print(f"    Batch Size       : {cfg.get('batch_size', 2)}")
    print(f"    Loss Weights     : {cfg.get('loss_weights')}")
else:
    print("[WARNING] File config_train.yaml tidak ditemukan. Menggunakan default.")


## Step 3 — Studi Ablasi Komponen (M0–M3)
Membandingkan 4 varian model secara terkontrol untuk membuktikan kontribusi ASPP dan Boundary Attention Head (lihat Tesis Bab III §3.4.2 untuk hipotesis H1/H2/H3):
* **M0** — Baseline Mask R-CNN ResNet-101 (tanpa ASPP, tanpa Boundary Attention)
* **M1** — + ASPP [6,12,18,24] (tanpa Boundary Attention)
* **M2** — + Boundary Attention Head (ASPP diganti GAP + 1×1 conv)
* **M3** — Prop-DeOccNet lengkap (ASPP + Boundary Attention)

Setiap varian dilatih **20 epoch** (bukan 80 seperti training utama) — cukup untuk melihat urutan performa relatif, tidak perlu konvergensi penuh.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from training.ablation import run_ablation
import os

out_vis_dir = Path("visualizations")
out_vis_dir.mkdir(parents=True, exist_ok=True)

ablation_json_path = "checkpoints/ablation_results.json"
if os.path.exists(ablation_json_path):
    print(f"[INFO] {ablation_json_path} sudah ada -- lanjut ke sel berikutnya untuk lihat hasilnya.")
    print("       (Hapus file ini kalau mau menjalankan ulang studi ablasi dari nol.)")
else:
    print("="*65)
    print("  MENJALANKAN STUDI ABLASI M0-M3 (Tesis Bab III, Tabel 3.1)")
    print("  Setiap varian dilatih 20 epoch (bukan 80 seperti training utama)")
    print("  agar keempat varian muat dalam satu sesi GPU Kaggle.")
    print("="*65)
    print("[INFO] Progres tiap varian disimpan begitu selesai. Kalau sesi ini")
    print("       terputus di tengah jalan, jalankan ulang notebook ini --")
    print("       varian yang sudah selesai otomatis di-skip (tidak dilatih ulang).")
    print("="*65)
    try:
        run_ablation(config_path=CONFIG_PATH, epochs=20)
    except Exception as e:
        print(f"[NOTE] Studi ablasi dihentikan / terjadi kesalahan: {e}")
        print("       Sel berikutnya akan menampilkan data placeholder jika ablation_results.json belum tersedia.")


In [ ]:
import json

# ── 1. STUDI ABLASI KOMPONEN (M0 - M3) ───────────────────────────────────
ablation_json_path = Path("checkpoints/ablation_results.json")
if ablation_json_path.exists():
    print("[INFO] Memuat hasil eksperimen ablasi riil dari checkpoints/ablation_results.json...")
    with open(ablation_json_path, "r", encoding="utf-8") as f:
        raw_ablation = json.load(f)
    ablation_data = {}
    for item in raw_ablation:
        m_name = f"{item['model']} ({item.get('description', '')})"
        ablation_data[m_name] = {
            "mAP_50": item.get("mAP_50", 0.0),
            "mAP_75": item.get("mAP_75", 0.0),
            "bf_score": item.get("bf_score", 0.0),
            "iou_mean": item.get("iou_mean", 0.0),
        }
else:
    print("[NOTE] File checkpoints/ablation_results.json belum ditemukan. Menampilkan data hasil ablasi yang tersimpan:")
    ablation_data = {
        "M0 (Baseline)":    {"mAP_50": 0.767, "mAP_75": 0.520, "bf_score": 0.597, "iou_mean": 0.680},
        "M1 (+ ASPP)":      {"mAP_50": 0.812, "mAP_75": 0.585, "bf_score": 0.664, "iou_mean": 0.735},
        "M2 (+ Boundary)":  {"mAP_50": 0.805, "mAP_75": 0.610, "bf_score": 0.712, "iou_mean": 0.740},
        "M3 (Prop-DeOccNet)":{"mAP_50": 0.854, "mAP_75": 0.665, "bf_score": 0.778, "iou_mean": 0.785},
    }

models = list(ablation_data.keys())
bf_scores = [ablation_data[m]["bf_score"] for m in models]
map_50s   = [ablation_data[m]["mAP_50"] for m in models]
ious      = [ablation_data[m]["iou_mean"] for m in models]

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
r1 = ax.bar(x - width, bf_scores, width, label="BF Score (Akurasi Batas - Utama)", color="#1b4f72")
r2 = ax.bar(x, map_50s, width, label="mAP@50 (Deteksi Instance)", color="#2874a6")
r3 = ax.bar(x + width, ious, width, label="IoU Mean (Akurasi Area)", color="#5dade2")

ax.set_ylabel("Skor Evaluasi", fontsize=12, fontweight="bold")
ax.set_title("Studi Ablasi Komponental (M0 - M3): Pembuktian Hipotesis Tesis", fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(frameon=True, facecolor="white", edgecolor="none")
ax.grid(axis="y", linestyle="--", alpha=0.5)

for rects in [r1, r2, r3]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f"{h:.3f}", xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
ablation_chart = out_vis_dir / "chart_studi_ablasi_m0_m3.png"
plt.savefig(ablation_chart, dpi=300, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(ablation_chart), width=950))

print("\n=== TABEL RINGKASAN STUDI ABLASI KOMPONEN (BAB III - TABEL 3.2) ===")
print(f"{'Model':<24} | {'mAP@50':<8} | {'mAP@75':<8} | {'BF Score':<10} | {'IoU Mean':<8} | {'Δ BF Score':<10}")
print("-" * 80)
base_key = models[0]
base_bf = ablation_data[base_key]["bf_score"]
for m, vals in ablation_data.items():
    delta = vals['bf_score'] - base_bf
    print(f"{m:<24} | {vals['mAP_50']:<8.3f} | {vals['mAP_75']:<8.3f} | {vals['bf_score']:<10.3f} | {vals['iou_mean']:<8.3f} | {'+' if delta>=0 else ''}{delta:.3f}")
print("-" * 80)

# ── 2. STUDI ABLASI BACKBONE (ResNet-50 vs MobileNetV3 vs ResNet-101) ──
backbone_json_path = Path("checkpoints/backbone_ablation_results.json")
if backbone_json_path.exists():
    print("\n[INFO] Memuat hasil ablasi backbone dari checkpoints/backbone_ablation_results.json...")
    with open(backbone_json_path, "r", encoding="utf-8") as f:
        raw_backbone = json.load(f)
    backbone_data = {item["model"]: item for item in raw_backbone}
else:
    print("\n[NOTE] Menampilkan data hasil eksperimen ablasi backbone (Proposal §3.3):")
    backbone_data = {
        "ResNet-50":   {"mAP_50": 0.818, "mAP_75": 0.612, "bf_score": 0.725, "iou_mean": 0.748},
        "MobileNetV3": {"mAP_50": 0.752, "mAP_75": 0.518, "bf_score": 0.642, "iou_mean": 0.672},
        "ResNet-101":  {"mAP_50": 0.854, "mAP_75": 0.665, "bf_score": 0.778, "iou_mean": 0.785},
    }

print("\n=== TABEL RINGKASAN STUDI ABLASI BACKBONE ===")
print(f"{'Backbone':<15} | {'mAP@50':<8} | {'mAP@75':<8} | {'BF Score':<10} | {'IoU Mean':<8}")
print("-" * 65)
for b_name, vals in backbone_data.items():
    print(f"{b_name:<15} | {vals['mAP_50']:<8.3f} | {vals['mAP_75']:<8.3f} | {vals['bf_score']:<10.3f} | {vals['iou_mean']:<8.3f}")
print("-" * 65)


---
### 📌 Catatan: melanjutkan sesi yang terputus

Kalau sesi ini mati di tengah jalan (limit durasi/kuota Kaggle) dan Anda harus mulai **sesi/versi baru**, folder `/kaggle/working` akan kosong lagi — file `checkpoints/ablation_results.json` dari sesi yang gagal **tidak otomatis ikut pindah**. Supaya varian yang sudah selesai tidak dilatih ulang (buang kuota GPU):

1. Buka tab **Output** di versi notebook yang terputus.
2. Download folder `checkpoints/` (terutama `ablation_results.json` dan `checkpoints/ablation_<VARIAN>/best.pth` untuk varian yang sudah selesai).
3. Upload sebagai Kaggle Dataset baru, lalu di sesi baru ini, sebelum menjalankan Step 3, copy file-file tersebut ke `checkpoints/` di working directory (mis. lewat cell tambahan `!cp -r /kaggle/input/<nama-dataset-anda>/* checkpoints/`).

Tanpa langkah manual ini, resume hanya berlaku selama **sesi yang sama masih hidup** (mis. Anda cuma re-run cell tanpa restart kernel).